# Semana 4 · Parte 1 — Funciones y gráficas

**Señales y Sistemas** · 2026-2
Universidad Católica San Pablo · Facultad de Ingeniería
Ing. Electrónica y de Telecomunicaciones
Jefe de práctica: Ing. Jeanpier Ancori Sánchez

---

## Objetivos

Al terminar esta sesión el alumno será capaz de:

1. Escribir funciones en Python que devuelvan una señal a partir de sus parámetros.
2. Construir un vector de tiempo con `linspace` y con `arange`, y saber cuándo conviene cada uno.
3. Elegir entre `plot` y `stem` según la señal sea continua o discreta.
4. Rotular una gráfica de forma completa: título, ejes con unidades, rejilla y leyenda.
5. Relacionar la frecuencia de muestreo $f_m$, el período de muestreo $T_s$ y el número
   de muestras $N$ de un registro.

---
## 1. Preparación del entorno

Todo el material usa dos librerías: **NumPy** para el cálculo numérico y **Matplotlib**
para graficar. Se importan una sola vez, al inicio del notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Rejilla siempre visible en todas las figuras.
plt.rcParams["axes.grid"] = True

---
## 2. El vector de tiempo

Antes de generar cualquier señal hay que decidir **en qué instantes** se va a evaluar.
Ese conjunto de instantes es el *vector de tiempo*. NumPy ofrece dos formas de construirlo:

| Función | Se le indica | Úsala cuando |
|---|---|---|
| `np.linspace(inicio, fin, N)` | el **número de puntos** $N$ | quieres una curva suave para dibujar |
| `np.arange(inicio, fin, paso)` | el **paso** entre puntos | el paso es el período de muestreo $T_s$ |

Ambas describen el mismo intervalo, pero se parametrizan distinto:

$$
T_s = \frac{t_f - t_0}{N - 1}
\qquad\Longleftrightarrow\qquad
N = \frac{t_f - t_0}{T_s} + 1
$$

**Cuidado:** `arange` **no incluye** el extremo final; `linspace` sí lo incluye.

In [ ]:
# Con linspace: se pide el número de puntos.
t_linspace = np.linspace(0, 1, 11)

# Con arange: se pide el paso.
t_arange = np.arange(0, 1, 0.1)

print("linspace ->", t_linspace)
print("arange   ->", t_arange)

print("linspace tiene", len(t_linspace), "puntos e incluye t = 1")
print("arange   tiene", len(t_arange), "puntos y NO incluye t = 1")

---
## 3. Funciones en Python

Una función encapsula un cálculo para poder repetirlo cambiando solo sus parámetros.
En vez de copiar y pegar la misma expresión cada vez que se necesita una senoide,
se escribe una vez y se llama tantas veces como haga falta.

Se define con `def`, y `return` entrega el resultado:

$$
x(t) = A\,\sin(2\pi f t)
\qquad\qquad
x(t) = A\,\cos(2\pi f t)
$$

En este curso **todo se construirá con funciones**. Vale la pena acostumbrarse desde ahora.

In [ ]:
def seno(t, f, A=1.0):
    """Senoide de amplitud A y frecuencia f, evaluada en los instantes t."""
    x = A * np.sin(2 * np.pi * f * t)
    return x


def coseno(t, f, A=1.0):
    """Cosenoide de amplitud A y frecuencia f, evaluada en los instantes t."""
    x = A * np.cos(2 * np.pi * f * t)
    return x

### 3.1. El orden de los argumentos importa

Python asigna los argumentos **por posición**: el primer valor que se escribe va al primer
parámetro, el segundo al segundo, y así sucesivamente. Si se intercambian dos, la función
**no da error** — simplemente calcula otra cosa, y el fallo pasa desapercibido hasta que
la gráfica sale rara.

Por eso en este material se adopta una convención fija: **el tiempo va siempre primero**.

Para los demás parámetros conviene escribir el nombre, así:

```python
seno(t, f=2, A=1)
```

Esa forma no depende del orden y además se lee sola.

El parámetro `A=1.0` de la definición es un **valor por defecto**: si no se escribe la
amplitud al llamar la función, vale 1.

In [ ]:
t = np.linspace(0, 1, 500)

# Las tres llamadas siguientes son equivalentes y correctas:
x_a = seno(t, 2)              # por posición
x_b = seno(t, f=2)            # la frecuencia, con su nombre
x_c = seno(t, f=2, A=1.0)     # todo con nombre  <- la más legible

print("¿x_a es igual a x_b?", np.allclose(x_a, x_b))
print("¿x_b es igual a x_c?", np.allclose(x_b, x_c))

# En cambio, esta llamada intercambia t y f. No falla, pero calcula otra señal:
x_mal = seno(2, t)            # <- ERROR SILENCIOSO
print("¿seno(2, t) es igual a seno(t, 2)?", np.allclose(x_mal, x_a))

---
## 4. Graficar una señal

Matplotlib ofrece dos representaciones que usaremos constantemente:

| Función | Dibuja | Representa |
|---|---|---|
| `plt.plot(t, x)` | puntos unidos por segmentos | una señal **continua** $x(t)$ |
| `plt.stem(n, x)` | una línea vertical por muestra | una señal **discreta** $x[n]$ |

El criterio es conceptual, no estético: `stem` afirma que la señal **solo existe** en esos
instantes, mientras que `plot` sugiere que existe también entre ellos.

En la práctica `stem` deja de leerse bien por encima de unas 40 muestras; para registros
largos se usa `plot` aunque la señal sea discreta.

Para poner dos gráficas en la misma figura se usa `plt.subplot(filas, columnas, posición)`:
selecciona el panel donde se dibuja a partir de ahí.

In [ ]:
f = 2.0          # frecuencia de la señal, Hz
fm = 20.0        # frecuencia de muestreo, Hz

t_denso = np.linspace(0, 1, 1000)      # para dibujar la señal continua
t_muestras = np.arange(0, 1, 1 / fm)   # instantes de muestreo

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(t_denso, seno(t_denso, f=f))
plt.title(f"Señal continua: senoide de {f} Hz")
plt.ylabel("Amplitud [V]")

plt.subplot(2, 1, 2)
plt.stem(t_muestras, seno(t_muestras, f=f))
plt.title(f"Señal discreta: {len(t_muestras)} muestras a fm = {fm} Hz")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud [V]")

plt.tight_layout()
plt.show()

### 4.1. Rotulado de las gráficas

> **Una gráfica sin rótulos no comunica nada.** En los laboratorios de este curso, toda
> figura debe llevar **título**, **ejes rotulados con sus unidades**, **rejilla** y, si hay
> más de una curva, **leyenda**. Una figura sin rotular se califica con **cero**.

No es una formalidad: sin unidades en los ejes, el lector no puede saber si el eje horizontal
está en segundos o en milisegundos, y la gráfica resulta indistinguible de otra mil veces
más rápida.

In [ ]:
t = np.linspace(0, 1, 500)
x = seno(t, f=3, A=2)

plt.figure(figsize=(12, 4))

# --- Cómo NO se hace -------------------------------------------------
plt.subplot(1, 2, 1)
plt.plot(t, x)
plt.title("Incorrecto: sin unidades ni información")

# --- Cómo se hace ----------------------------------------------------
plt.subplot(1, 2, 2)
plt.plot(t, x, label="x(t) = 2 sen(2 pi 3 t)")
plt.title("Correcto: senoide de 3 Hz y amplitud 2 V")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud [V]")
plt.legend()

plt.tight_layout()
plt.show()

---
## 5. Muestreo

Muestrear es quedarse con el valor de la señal cada $T_s$ segundos:

$$
x[n] = x(n\,T_s), \qquad n = 0, 1, 2, \dots, N-1
$$

donde el período de muestreo y la frecuencia de muestreo son recíprocos:

$$
T_s = \frac{1}{f_m}
$$

El número de muestras que caben en un registro de duración $t_f$ es:

$$
\boxed{\,N = f_m \cdot t_f\,}
$$

**Este es el punto donde más se falla.** El número de muestras se calcula con la **duración
total del registro**, no con el período de la señal. Si el registro dura dos períodos, hay
que multiplicar $f_m$ por esos dos períodos completos; usar solo uno deja la mitad de las
muestras, y la frecuencia de muestreo real acaba siendo la mitad de la que se creía tener.

In [ ]:
f = 1.0                  # frecuencia de la señal, Hz
A = 2.0                  # amplitud, V
fm = 8.0                 # frecuencia de muestreo, Hz

periodo = 1 / f          # período de la señal, s
tf = 2 * periodo         # el registro dura DOS períodos
Ts = 1 / fm              # período de muestreo, s
N = int(fm * tf)         # <- con tf, no con 'periodo'

print("Período de la señal   T  =", periodo, "s")
print("Duración del registro tf =", tf, "s")
print("Período de muestreo   Ts =", Ts, "s")
print("Número de muestras    N  =", N)

t_denso = np.linspace(0, tf, 1000)
t_muestras = np.arange(0, tf, Ts)

plt.figure(figsize=(10, 4))
plt.plot(t_denso, seno(t_denso, f=f, A=A), "--", label="x(t) continua")
plt.stem(t_muestras, seno(t_muestras, f=f, A=A), label="x[n] muestreada")
plt.title(f"Senoide de {f} Hz muestreada a {fm} Hz durante {tf} s: {N} muestras")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud [V]")
plt.legend()
plt.show()

### 5.1. Efecto de la frecuencia de muestreo

Cuantas menos muestras por período, peor queda descrita la señal.

Para comparar tres casos sin repetir el mismo código tres veces, escribimos **una función
que dibuja un panel** y la llamamos tres veces con distinta frecuencia de muestreo. Es
exactamente para esto que sirven las funciones.

In [ ]:
def dibujar_muestreo(f, fm, posicion):
    """Dibuja, en el panel indicado, una senoide de f Hz y sus muestras a fm Hz."""
    t_denso = np.linspace(0, 1, 1000)
    t_muestras = np.arange(0, 1, 1 / fm)
    muestras_por_periodo = fm / f

    plt.subplot(3, 1, posicion)
    plt.plot(t_denso, seno(t_denso, f=f), "--", label="x(t)")
    plt.stem(t_muestras, seno(t_muestras, f=f), label="x[n]")
    plt.title(f"fm = {fm} Hz  ->  {muestras_por_periodo} muestras por período")
    plt.ylabel("Amplitud [V]")
    plt.legend(loc="upper right")

In [ ]:
plt.figure(figsize=(10, 8))

dibujar_muestreo(f=2, fm=40, posicion=1)
dibujar_muestreo(f=2, fm=10, posicion=2)
dibujar_muestreo(f=2, fm=3, posicion=3)

plt.xlabel("Tiempo [s]")
plt.tight_layout()
plt.show()

Fíjate en el tercer panel: con $f_m = 3$ Hz se toman menos de dos muestras por período, y
las muestras ya no dibujan la señal original sino una mucho más lenta. Ese fenómeno —el
**aliasing**— es lo que motiva el teorema de muestreo que se verá más adelante.

---
## 6. Superposición de senoides

Sumar señales es la operación más común del curso. Si

$$
x_1(t) = A_1\sin(2\pi f_1 t), \qquad
x_2(t) = A_2\sin(2\pi f_2 t), \qquad
x_3(t) = A_3\sin(2\pi f_3 t)
$$

entonces la suma es:

$$
y(t) = x_1(t) + x_2(t) + x_3(t)
$$

Como `seno` devuelve un arreglo de NumPy, sumar las tres señales es simplemente escribir
`x1 + x2 + x3`: NumPy suma **muestra a muestra**, sin necesidad de recorrer nada.

In [ ]:
def suma_de_tres_senos(t, f1, f2, f3):
    """Suma de tres senoides de frecuencias f1, f2, f3 y amplitudes 2, 3 y 4."""
    x1 = seno(t, f=f1, A=2)
    x2 = seno(t, f=f2, A=3)
    x3 = seno(t, f=f3, A=4)
    y = x1 + x2 + x3
    return y


def dibujar_suma(t, f1, f2, f3, posicion):
    """Dibuja la suma de las tres senoides en el panel indicado."""
    plt.subplot(3, 1, posicion)
    plt.plot(t, suma_de_tres_senos(t, f1, f2, f3))
    plt.title(f"y(t) con f1 = {f1} Hz, f2 = {f2} Hz, f3 = {f3} Hz")
    plt.ylabel("Amplitud [V]")

In [ ]:
t = np.linspace(0, 2, 1000)

plt.figure(figsize=(10, 8))

dibujar_suma(t, 2, 3, 4, posicion=1)
dibujar_suma(t, 4, 6, 8, posicion=2)
dibujar_suma(t, 10, 20, 30, posicion=3)

plt.xlabel("Tiempo [s]")
plt.tight_layout()
plt.show()

---
## 7. Ejercicios propuestos

1. Escribe una función `suma_seno_coseno(t, f)` que devuelva
   $y(t) = \sin(2\pi f t) + \cos^2(2\pi f t)$ y grafícala para $f = 30$ Hz en el
   intervalo $t \in [0,\ 0{,}2]$ s. Rotula la figura por completo.

2. Genera una cosenoide de 2 Hz y muéstrala junto con sus muestras a $f_m = 100$ Hz
   y a $f_m = 60$ Hz en la misma figura. Imprime los dos períodos de muestreo y
   comprueba que $T_s = 1/f_m$.

3. ¿Cuántas muestras tiene un registro de 3 s adquirido a 44 100 Hz? Calcúlalo primero
   a mano y confírmalo con `len()`.

4. Modifica `suma_de_tres_senos` para que las amplitudes también sean parámetros de la
   función, en lugar de estar fijas en 2, 3 y 4.

5. Grafica $y(t) = \sin(2\pi t) + \tfrac{1}{3}\sin(6\pi t) + \tfrac{1}{5}\sin(10\pi t)$
   sobre $t \in [0,\ 2]$ s. ¿A qué forma de onda se va pareciendo conforme se añaden
   más términos impares?